# 📨 W7: Slack 생산 알림 자동화
**hexa-1 — Week 7** | ⏱️ 60분 | 🎯 KPI 요약 → Slack 자동 발송

> 💡 Slack 웹훅이 없어도 콘솔 출력으로 실습 가능

## Step 0: 환경 설정

In [ ]:
!pip install -q pandas requests openpyxl
import pandas as pd, requests, json
print('✅ 준비 완료')


## Step 1: 설정 입력 (✏️ 여기만 수정)

In [ ]:
CONFIG = {
    'factory_name': '✏️ [교육 기업명]',
    'webhook_url': '',  # Slack Incoming Webhook URL (없으면 콘솔 출력)
    'channel': '#생산관리',
}
# KPI 데이터 (실제 데이터로 교체 가능)
KPI_DATA = {
    '날짜': '2026-03-04',
    '총생산량': 950,
    '불량수': 12,
    'OEE': 87.3,
    '가동률': 94.5,
}
print('✅ 설정 완료')


## Step 2: Slack 메시지 생성

In [ ]:
def build_slack_message(kpi: dict, factory: str) -> dict:
    defect_rate = round(kpi['불량수'] / kpi['총생산량'] * 100, 2)
    text = (
        f'🏭 *{factory} 일일 생산 보고* ({kpi["날짜"]})
'
        f'📦 총생산량: {kpi["총생산량"]:,}개
'
        f'⚠️ 불량수: {kpi["불량수"]}개 ({defect_rate}%)
'
        f'⚙️ OEE: {kpi["OEE"]}%  ·  가동률: {kpi["가동률"]}%
'
        f'{"✅ 목표 달성" if kpi["OEE"] >= 85 else "🔄 OEE 개선 필요"}'
    )
    return {'text': text, 'channel': CONFIG['channel']}

msg = build_slack_message(KPI_DATA, CONFIG['factory_name'])
print('=== 생성된 Slack 메시지 ===')
print(msg['text'])


## Step 3: Slack 발송 (웹훅 없으면 자동 스킵)

In [ ]:
def send_to_slack(message: dict, webhook_url: str) -> None:
    if not webhook_url:
        print('⏭️ Webhook URL 없음 → Slack 전송 생략 (콘솔 출력만)')
        return
    try:
        resp = requests.post(webhook_url, json=message, timeout=5)
        if resp.status_code == 200:
            print('✅ Slack 발송 성공!')
        else:
            print(f'⚠️ 발송 실패: {resp.status_code} {resp.text}')
    except Exception as e:
        print(f'오류: {e}')

send_to_slack(msg, CONFIG['webhook_url'])
print('\n👉 실제 Slack Webhook을 사용하려면:')
print('   1. slack.com → 앱 관리 → Incoming Webhooks 추가')
print('   2. webhook_url에 발급받은 URL 입력')


## Step 4: 배치 처리 (여러 날짜 한번에)

In [ ]:
import numpy as np
np.random.seed(1)
daily_data = pd.DataFrame({
    '날짜': pd.date_range('2026-03-01', periods=7).strftime('%Y-%m-%d'),
    '총생산량': np.random.randint(880, 980, 7),
    '불량수': np.random.randint(5, 20, 7),
    'OEE': np.random.uniform(82, 92, 7).round(1),
    '가동률': np.random.uniform(90, 97, 7).round(1),
})

print('=== 주간 생산 요약 Slack 메시지 ===')
for _, row in daily_data.iterrows():
    msg = build_slack_message(row.to_dict(), CONFIG['factory_name'])
    print(f'--- {row["날짜"]} ---')
    print(msg['text'])
    print()
print(f'✅ {len(daily_data)}일치 메시지 생성 완료')


---
## 🔥 확장 과제
1. 실제 Slack Webhook URL을 받아서 발송해보기
2. 불량률이 3% 초과 시 '@channel 긴급' 알림 추가
3. `auto_report_generator.py`와 연결해서 Excel→Slack 파이프라인 완성

In [ ]:
# === Gemini AI 분석 (자동 추가됨) ===
!pip install -q google-generativeai
try:
    from google.colab import userdata; _api=userdata.get('GEMINI_API_KEY')
except: _api=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=_api)
_model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ Gemini 분석 준비')


## 🤖 AI 인사이트 분석 (Gemini)

In [ ]:
_p=f"""제조업 AI 컨설턴트로서 KPI 경보 상황를 분석하세요.
핵심 인사이트 3가지 + 즉시 개선 액션. 마크다운."""
_resp=_model.generate_content(_p)
print(_resp.text)
